# DQN

使用神经网络近似动作价值函数的 Deep Q-Network。

## 核心思想

表格型 Q-learning 为每个状态—动作对保存一个 Q 值，但在状态空间较大或连续时无法维护完整表格。DQN（Deep Q-Network）使用参数为 θ 的神经网络 Qθ(s,a) 近似动作价值函数，网络一次输出当前状态下所有离散动作的价值。

智能体使用 ε-greedy 策略：以概率 ε 随机探索，否则选择当前 Q 值最大的动作。训练期间让 ε 从较大值逐渐衰减，可以在早期充分探索、后期更多利用已经学到的策略。

## TD 目标与损失

对于转移 (sₜ,aₜ,rₜ,sₜ₊₁)，一步 Q-learning 目标为

$$
y_t=r_t+\gamma(1-d_t)\max_{a'}Q_{\theta^-}(s_{t+1},a').
$$

其中 dₜ 表示真正的终止状态，θ⁻ 是目标网络参数。在线网络最小化均方 TD 误差：

$$
L(\theta)=\mathbb{E}\left[\left(Q_\theta(s_t,a_t)-y_t\right)^2\right].
$$

## 两个稳定化技巧

- **经验回放**：把环境转移保存到缓冲区，再随机采样小批量训练，降低相邻样本之间的相关性并重复利用数据。
- **目标网络**：使用一份延迟更新的网络计算 TD 目标，避免预测值与训练目标同时快速变化。每隔若干次梯度更新，将在线网络参数复制到目标网络。

## 代码实现

### 导入依赖与环境兼容函数

In [ ]:
import math
import random
from collections import deque

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F

try:
    import gymnasium as gym
except ImportError:
    import gym


def reset_env(env, seed=None):
    result = env.reset(seed=seed)
    return result[0] if isinstance(result, tuple) else result


def step_env(env, action):
    result = env.step(action)
    if len(result) == 5:
        next_state, reward, terminated, truncated, info = result
        return next_state, reward, terminated or truncated, terminated, info
    next_state, reward, done, info = result
    return next_state, reward, done, done, info

### 经验回放缓冲区

In [ ]:
class ReplayBuffer:
    def __init__(self, capacity):
        self.buffer = deque(maxlen=capacity)

    def add(self, state, action, reward, next_state, terminal):
        self.buffer.append((state, action, reward, next_state, terminal))

    def sample(self, batch_size):
        transitions = random.sample(self.buffer, batch_size)
        states, actions, rewards, next_states, terminals = zip(*transitions)
        return (
            np.asarray(states, dtype=np.float32),
            np.asarray(actions, dtype=np.int64),
            np.asarray(rewards, dtype=np.float32),
            np.asarray(next_states, dtype=np.float32),
            np.asarray(terminals, dtype=np.float32),
        )

    def __len__(self):
        return len(self.buffer)

### Q 网络与 DQN 智能体

在线 Q 网络负责选择动作和学习；目标 Q 网络只负责构造较稳定的 TD 目标。

In [ ]:
class QNet(torch.nn.Module):
    def __init__(self, state_dim, hidden_dim, action_dim):
        super().__init__()
        self.fc1 = torch.nn.Linear(state_dim, hidden_dim)
        self.fc2 = torch.nn.Linear(hidden_dim, action_dim)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        return self.fc2(x)


class DQN:
    def __init__(self, state_dim, hidden_dim, action_dim, learning_rate,
                 gamma, epsilon_start, epsilon_end, epsilon_decay,
                 target_update, device):
        self.q_net = QNet(state_dim, hidden_dim, action_dim).to(device)
        self.target_q_net = QNet(state_dim, hidden_dim, action_dim).to(device)
        self.target_q_net.load_state_dict(self.q_net.state_dict())
        self.optimizer = torch.optim.Adam(self.q_net.parameters(), lr=learning_rate)
        self.action_dim = action_dim
        self.gamma = gamma
        self.epsilon_start = epsilon_start
        self.epsilon_end = epsilon_end
        self.epsilon_decay = epsilon_decay
        self.target_update = target_update
        self.device = device
        self.environment_steps = 0
        self.update_steps = 0

    @property
    def epsilon(self):
        return self.epsilon_end + (
            self.epsilon_start - self.epsilon_end
        ) * math.exp(-self.environment_steps / self.epsilon_decay)

    @torch.no_grad()
    def take_action(self, state):
        epsilon = self.epsilon
        self.environment_steps += 1
        if random.random() < epsilon:
            return random.randrange(self.action_dim)
        state = torch.as_tensor(
            state, dtype=torch.float32, device=self.device
        ).unsqueeze(0)
        return self.q_net(state).argmax(dim=1).item()

    def update(self, transitions):
        states, actions, rewards, next_states, terminals = transitions
        states = torch.as_tensor(states, dtype=torch.float32, device=self.device)
        actions = torch.as_tensor(actions, dtype=torch.long, device=self.device).unsqueeze(1)
        rewards = torch.as_tensor(rewards, dtype=torch.float32, device=self.device).unsqueeze(1)
        next_states = torch.as_tensor(next_states, dtype=torch.float32, device=self.device)
        terminals = torch.as_tensor(terminals, dtype=torch.float32, device=self.device).unsqueeze(1)

        q_values = self.q_net(states).gather(1, actions)
        with torch.no_grad():
            max_next_q_values = self.target_q_net(next_states).max(dim=1, keepdim=True).values
            td_targets = rewards + self.gamma * max_next_q_values * (1 - terminals)

        loss = F.mse_loss(q_values, td_targets)
        self.optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(self.q_net.parameters(), 10.0)
        self.optimizer.step()

        self.update_steps += 1
        if self.update_steps % self.target_update == 0:
            self.target_q_net.load_state_dict(self.q_net.state_dict())
        return loss.item()

### 训练

先让回放缓冲区积累一定数量的转移；之后每与环境交互一步，就从缓冲区随机采样一个批次更新 DQN。对于 Gymnasium 的时间上限截断，本实现结束当前 episode，但仍允许 TD 目标从下一状态自举。

In [ ]:
learning_rate = 2e-3
num_episodes = 500
hidden_dim = 128
gamma = 0.98
epsilon_start = 1.0
epsilon_end = 0.01
epsilon_decay = 2000
target_update = 100
buffer_capacity = 10000
minimal_size = 500
batch_size = 64
seed = 0
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

env_name = 'CartPole-v1'
env = gym.make(env_name)
env.action_space.seed(seed)
state_dim = env.observation_space.shape[0]
action_dim = env.action_space.n
replay_buffer = ReplayBuffer(buffer_capacity)
agent = DQN(
    state_dim, hidden_dim, action_dim, learning_rate, gamma,
    epsilon_start, epsilon_end, epsilon_decay, target_update, device
)

return_list = []
loss_list = []
for episode in range(num_episodes):
    state = reset_env(env, seed + episode)
    episode_return = 0.0
    done = False
    while not done:
        action = agent.take_action(state)
        next_state, reward, done, terminal, _ = step_env(env, action)
        replay_buffer.add(state, action, reward, next_state, terminal)
        state = next_state
        episode_return += reward

        if len(replay_buffer) >= minimal_size:
            loss = agent.update(replay_buffer.sample(batch_size))
            loss_list.append(loss)

    return_list.append(episode_return)
    if (episode + 1) % 50 == 0:
        recent_return = np.mean(return_list[-50:])
        print(
            f'Episode {episode + 1:3d} | '
            f'mean return {recent_return:7.2f} | '
            f'epsilon {agent.epsilon:.3f}'
        )

env.close()

In [ ]:
def moving_average(values, window_size):
    values = np.asarray(values, dtype=np.float32)
    if len(values) < window_size:
        return values
    kernel = np.ones(window_size, dtype=np.float32) / window_size
    return np.convolve(values, kernel, mode='valid')


plt.figure(figsize=(10, 4))
plt.plot(return_list, alpha=0.25, label='Episode return')
window_size = 25
smoothed_returns = moving_average(return_list, window_size)
plt.plot(
    np.arange(window_size - 1, len(return_list)),
    smoothed_returns,
    label=f'{window_size}-episode moving average'
)
plt.xlabel('Episodes')
plt.ylabel('Returns')
plt.title(f'DQN on {env_name}')
plt.legend()
plt.show()

## 与策略梯度方法的区别

DQN 直接学习每个离散动作的价值，并通过最大 Q 值选择动作；PPO、GRPO 等方法则直接优化参数化策略。DQN 可以借助经验回放重复使用历史数据，样本利用率较高，但基本形式只适用于离散动作空间。连续动作任务通常使用 DDPG、TD3 或 SAC 等 Actor-Critic 方法。